<a href="https://colab.research.google.com/github/yiyu-chen-labs/llm-from-scratch/blob/main/week10_Chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chunking 筆記 — Week 10 (RAG 檢索前置)

## chunking 是什麼、為什麼要
- chunking = 把長文件切成小塊(chunk)。
- 為什麼:LLM 塞不下整份長文件,且只需要相關的部分。
  所以先切塊,檢索時只撈相關的幾塊。
- chunk 就是「檢索的對象」—— 之前用 JQaRA 時,那些段落是資料集
  幫我切好的,我沒切過;Week 10 是第一次自己切原始長文件。

## 為什麼 chunking 是「失分點 / 賣點」
- 怎麼切,大大影響檢索準不準。多數人用最偷懶的固定長度硬切,
  會切斷語意(詞、句、完整的意思被拆到兩塊)。
- 切得聰明(不切斷語意)→ 檢索品質贏過別人 = 賣點。

## 今天做了什麼(手刻,沒用套件)

### 1. 固定長度 chunker(最陽春,會切爛)
    chunk_size = 100
    chunks = []
    for start in range(0, len(text), chunk_size):
        chunk = text[start:start+chunk_size]
        chunks.append(chunk)
- 素材:一段日文 Wikipedia 文章(Frank Frazetta),含標題/表格/雜訊。
- 結果:切成 11 塊。
- **親眼看到問題**:人名 "Frazzetta" 被切成 `Frazze`(chunk0尾)+ `tta`(chunk1頭),
  一個詞裂成兩塊 → 檢索時兩塊都撈不準,答案在文件裡卻找不到。

### 2. 加 overlap(重疊,補救切斷)
    chunk_size = 100
    overlap_size = 20
    chunks = []
    for start in range(0, len(text), chunk_size - overlap_size):  # 步長變小
        chunk = text[start:start+chunk_size]                       # 取的長度不變
        chunks.append(chunk)
- 關鍵關係式:**前進量(步長)= chunk_size - overlap**。
  前進量比 chunk_size 小 → 相鄰塊重疊。
- 結果:切成 15 塊(比 11 多),"Frazzetta" 在 chunk1 裡完整出現了 ✅
- **代價**:塊變多(重疊 = 冗餘 = 更多儲存/計算)。overlap 是取捨,不是免費。

## 今天最重要的領悟
- overlap「看起來」救回一個名字,但**不能憑一個例子就說它比較好**。
- 要證明「哪種切法真的好」,得用數字量 —— 就是 Week 9 學的 recall@k。
- 這就是「消融表(ablation table)」要做的事(見下)。

## 消融表(ablation table)是什麼 —— 下次要做
- ablation = 一次只改一個變數,看效果(控制變數實驗)。
- 消融表 = 列出「不同切法各自的 recall」,用數字比哪種好、好多少。
- 範例形狀:
  | 切法 | overlap | recall@1 | recall@5 | recall@10 |
  |------|---------|----------|----------|-----------|
  | 固定長度 | 0 | ? | ? | ? |
  | 固定長度 | 20 | ? | ? | ? |
  | 語意/結構切法 | 20 | ? | ? | ? |
- 這張表就是「賣點證據」:不是「我覺得 overlap 好」,而是
  「overlap 讓 recall@5 提升 X%」。

## 消融表現在做不了 —— 缺什麼(下次第一件事)
- 算 recall 需要「問題 + 答案標註(label)」。
- 現有資料的困境:
  - Wikipedia 文章:有原始長文件可切 ✅,但沒問題/沒 label ❌
  - JQaRA:有問題/label ✅,但段落已切好,沒東西給我切 ❌
- **卡點:需要一份「原始長文件 + 問題 + 答案」三者都有的資料集。**
- → 下次第一件事:找這樣一份資料集(而且做消融表要專門排 2-4 小時完整時段)。

## 今天做得出來的「觀察版」比較(不用 recall)
| 切法 | overlap | 切出幾塊 | Frazzetta | 觀察 |
|------|---------|----------|-----------|------|
| 固定長度 | 0 | 11 | ❌ 切成 Frazze+tta | 名字斷裂 |
| 固定長度 | 20 | 15 | ✅ chunk1 完整 | 救回,但塊變多 |

## 待辦 / 下一步
- [ ] 找「原始長文件 + 問題 + 答案」資料集(消融表前置)
- [ ] 做完整消融表:固定長度(overlap 0 vs 20)vs 語意切法,比 recall@k
- [ ] Week 10 其他項:metadata 注入、staleness 過期處理、語意/結構感知切法、
      chunker config 化
- [ ] 現成套件備查:LangChain text splitters、LlamaIndex(核心自己手刻過了,
      複雜的語意切法可用套件但要懂它在做什麼)

## 我現在在 RAG 的哪(給下次的自己)
- RAG = R(檢索)+ A(塞prompt)+ G(LLM生成)。
- 已做:R(檢索)+ recall@k 評估(Week 9)、手刻 chunker + overlap(Week 10 開頭)。
- 未做:消融表、語意切法、A + G(要 LLM,待接 API)。

In [1]:
text = """ 出典: フリー百科事典『ウィキペディア（Wikipedia）』
フランク・フラゼッタ
Frank Frazetta

フラゼッタ (1977)
生誕	Francesco Alfredo Frazzetta
1928年2月9日
アメリカ合衆国ニューヨークブルックリンシープスヘッドベイ
死没	2010年5月10日（82歳没）
アメリカ合衆国フロリダ州フォートマイヤーズ
教育	ブルックリン・アカデミー・オブ・ファインアーツ
著名な実績	イラストレーション、絵画
受賞
イラストレーター協会殿堂
世界ファンタジー大会生涯功労賞
アイズナー賞殿堂
テンプレートを表示
フランク・フラゼッタ（Frank Frazetta [frəˈzɛtə]、出生名 Francesco Alfredo Frazzetta、1928年2月9日 - 2010年5月10日）[1][2]はファンタジーやSFをテーマとする作品で知られるアメリカ人の画家・イラストレーター。コミック作画、ペーパーバック小説や雑誌の表紙、映画ポスターのような多様な媒体で活動した。筋肉質な英雄や肉感的な女性、怪物や戦闘場面の主題を、躍動的な人体表現と視線誘導の強い画面構成で描く作風で知られ、「剣と魔法」ジャンルのヴィジュアルイメージの形成に大きく寄与したとされる。日本でも1970年代半ば以降SF・ファンタジーや漫画・アニメ関係者を中心に受容された。

1940年代半ばに米国コミックブック作画家として出発し、SFヒーロー「バック・ロジャーズ」の表紙画などで評価を高めた。1950年代には新聞漫画の人気作『リル・アブナー』の作画スタッフを務めた。1960年代からはホラーコミック誌やエドガー・ライス・バローズの「ターザン」のようなペーパーバック書籍の表紙画を手がけはじめ、特にロバート・E・ハワードの「英雄コナン」の再刊装画で商業イラストレーターとしての名声を確立した。後に代表作と見なされる油彩作品の多くはこの時期に描かれている。その一つである『デス・ディーラー（英語版）』(1973) はレコードジャケットや雑誌表紙に用いられたほか、コミックなどの二次作品が作られ、軍のマスコットにも採用された。

1970年代後半以降は作家名そのものが販売力を持つ存在となり、多様な媒体に作品を提供したほか、複製画や関連商品の販売も自ら行った。アニメーション映画『ファイヤー&アイス』(1983) では原案提供のほか制作に携わった。1980年代半ばからは健康問題により制作活動に制約を受けたが、イラストレーションやファンタジー画、コミック作画の各分野で殿堂入りや生涯功労賞などの顕彰を受けた。没後は油彩作品『コナン（マンエイプ）』が1350万ドルで落札されるなど、ファンタジー画・コミック原画の取引額の記録を作っている。"""


In [2]:
text

' 出典: フリー百科事典『ウィキペディア（Wikipedia）』\nフランク・フラゼッタ\nFrank Frazetta\n\nフラゼッタ (1977)\n生誕\tFrancesco Alfredo Frazzetta\n1928年2月9日\nアメリカ合衆国ニューヨークブルックリンシープスヘッドベイ\n死没\t2010年5月10日（82歳没）\nアメリカ合衆国フロリダ州フォートマイヤーズ\n教育\tブルックリン・アカデミー・オブ・ファインアーツ\n著名な実績\tイラストレーション、絵画\n受賞\t\nイラストレーター協会殿堂\n世界ファンタジー大会生涯功労賞\nアイズナー賞殿堂\nテンプレートを表示\nフランク・フラゼッタ（Frank Frazetta [frəˈzɛtə]、出生名 Francesco Alfredo Frazzetta、1928年2月9日 - 2010年5月10日）[1][2]はファンタジーやSFをテーマとする作品で知られるアメリカ人の画家・イラストレーター。コミック作画、ペーパーバック小説や雑誌の表紙、映画ポスターのような多様な媒体で活動した。筋肉質な英雄や肉感的な女性、怪物や戦闘場面の主題を、躍動的な人体表現と視線誘導の強い画面構成で描く作風で知られ、「剣と魔法」ジャンルのヴィジュアルイメージの形成に大きく寄与したとされる。日本でも1970年代半ば以降SF・ファンタジーや漫画・アニメ関係者を中心に受容された。\n\n1940年代半ばに米国コミックブック作画家として出発し、SFヒーロー「バック・ロジャーズ」の表紙画などで評価を高めた。1950年代には新聞漫画の人気作『リル・アブナー』の作画スタッフを務めた。1960年代からはホラーコミック誌やエドガー・ライス・バローズの「ターザン」のようなペーパーバック書籍の表紙画を手がけはじめ、特にロバート・E・ハワードの「英雄コナン」の再刊装画で商業イラストレーターとしての名声を確立した。後に代表作と見なされる油彩作品の多くはこの時期に描かれている。その一つである『デス・ディーラー（英語版）』(1973) はレコードジャケットや雑誌表紙に用いられたほか、コミックなどの二次作品が作られ、軍のマスコットにも採用された。\n\n1970年代後半以降は作家名そのものが販売力を持つ存在となり、多様な媒体に作品を提供し

In [3]:
chunk_size = 100
chunks = []
for start in range(0, len(text), chunk_size):
    chunk = text[start:start+chunk_size]
    chunks.append(chunk)

In [4]:
print("總共切了", len(chunks), "塊\n")

for i, c in enumerate(chunks):
    print(f"--- chunk {i} ---")
    print(c)
    print()


總共切了 12 塊

--- chunk 0 ---
 出典: フリー百科事典『ウィキペディア（Wikipedia）』
フランク・フラゼッタ
Frank Frazetta

フラゼッタ (1977)
生誕	Francesco Alfredo Frazze

--- chunk 1 ---
tta
1928年2月9日
アメリカ合衆国ニューヨークブルックリンシープスヘッドベイ
死没	2010年5月10日（82歳没）
アメリカ合衆国フロリダ州フォートマイヤーズ
教育	ブルックリン・アカデミー

--- chunk 2 ---
・オブ・ファインアーツ
著名な実績	イラストレーション、絵画
受賞	
イラストレーター協会殿堂
世界ファンタジー大会生涯功労賞
アイズナー賞殿堂
テンプレートを表示
フランク・フラゼッタ（Frank 

--- chunk 3 ---
Frazetta [frəˈzɛtə]、出生名 Francesco Alfredo Frazzetta、1928年2月9日 - 2010年5月10日）[1][2]はファンタジーやSFをテーマとする作品

--- chunk 4 ---
で知られるアメリカ人の画家・イラストレーター。コミック作画、ペーパーバック小説や雑誌の表紙、映画ポスターのような多様な媒体で活動した。筋肉質な英雄や肉感的な女性、怪物や戦闘場面の主題を、躍動的な人体表

--- chunk 5 ---
現と視線誘導の強い画面構成で描く作風で知られ、「剣と魔法」ジャンルのヴィジュアルイメージの形成に大きく寄与したとされる。日本でも1970年代半ば以降SF・ファンタジーや漫画・アニメ関係者を中心に受容さ

--- chunk 6 ---
れた。

1940年代半ばに米国コミックブック作画家として出発し、SFヒーロー「バック・ロジャーズ」の表紙画などで評価を高めた。1950年代には新聞漫画の人気作『リル・アブナー』の作画スタッフを務めた

--- chunk 7 ---
。1960年代からはホラーコミック誌やエドガー・ライス・バローズの「ターザン」のようなペーパーバック書籍の表紙画を手がけはじめ、特にロバート・E・ハワードの「英雄コナン」の再刊装画で商業イラストレータ

--- chunk 8 ---
ーとしての名声を確立した。後に代表作と見なされる油彩作品の

In [5]:
#---OverLap---#
chunk_size = 100
overlap_size = 20
chunks = []
for start in range(0, len(text), chunk_size-overlap_size):
    chunk = text[start:start+chunk_size]
    chunks.append(chunk)


In [6]:
print("總共切了", len(chunks), "塊\n")

for i, c in enumerate(chunks):
    print(f"--- chunk {i} ---")
    print(c)
    print()


總共切了 15 塊

--- chunk 0 ---
 出典: フリー百科事典『ウィキペディア（Wikipedia）』
フランク・フラゼッタ
Frank Frazetta

フラゼッタ (1977)
生誕	Francesco Alfredo Frazze

--- chunk 1 ---
cesco Alfredo Frazzetta
1928年2月9日
アメリカ合衆国ニューヨークブルックリンシープスヘッドベイ
死没	2010年5月10日（82歳没）
アメリカ合衆国フロリダ州フォートマ

--- chunk 2 ---
没）
アメリカ合衆国フロリダ州フォートマイヤーズ
教育	ブルックリン・アカデミー・オブ・ファインアーツ
著名な実績	イラストレーション、絵画
受賞	
イラストレーター協会殿堂
世界ファンタジー大会生涯

--- chunk 3 ---
ーター協会殿堂
世界ファンタジー大会生涯功労賞
アイズナー賞殿堂
テンプレートを表示
フランク・フラゼッタ（Frank Frazetta [frəˈzɛtə]、出生名 Francesco Alfred

--- chunk 4 ---
出生名 Francesco Alfredo Frazzetta、1928年2月9日 - 2010年5月10日）[1][2]はファンタジーやSFをテーマとする作品で知られるアメリカ人の画家・イラストレー

--- chunk 5 ---
で知られるアメリカ人の画家・イラストレーター。コミック作画、ペーパーバック小説や雑誌の表紙、映画ポスターのような多様な媒体で活動した。筋肉質な英雄や肉感的な女性、怪物や戦闘場面の主題を、躍動的な人体表

--- chunk 6 ---
、怪物や戦闘場面の主題を、躍動的な人体表現と視線誘導の強い画面構成で描く作風で知られ、「剣と魔法」ジャンルのヴィジュアルイメージの形成に大きく寄与したとされる。日本でも1970年代半ば以降SF・ファン

--- chunk 7 ---
日本でも1970年代半ば以降SF・ファンタジーや漫画・アニメ関係者を中心に受容された。

1940年代半ばに米国コミックブック作画家として出発し、SFヒーロー「バック・ロジャーズ」の表紙画などで評価を

--- chunk 8 ---
バック・ロジャーズ」の表紙画などで評価を高めた。1950年